In [134]:
import numpy as np
import math
from graphviz import Digraph

What do we need to build?

We need to build an end to end training module which performes forward and backward pass on an MLP to find the best fit curve that fits the given data. 
For this, we'll need, a class that handles our data, does backward pass by differentiating. A neuron class which is the most atomic element of the MLP, a layer class and a training loop which fits the data perfectly

In [ ]:
# class that encapsulates our data
# does backprop as well

class Value:
    def __init__(self, data, _prev = (), _op = '', label = ''):
        self.data = data
        self._prev = _prev
        self.grad = 0
        self._op = _op
        self.label = label
        self._backward = lambda:None
    
    def __repr__(self):
        return f"Value(data = {self.data})"

    def __add__(self, other):
        if (isinstance(other, float) or isinstance(other, int)): other = Value(other)
        out = self.data + other.data
        out = Value(out, _prev = (self, other), _op = '+')
        def _backward():
            self.grad += 1.0*out.grad
            other.grad += 1.0*out.grad
        out._backward = _backward
        return out
    
    def __mul__(self, other):
        if (isinstance(other, float) or isinstance(other, int)): other = Value(other)
        out = self.data * other.data
        out = Value(out, _prev = (self, other), _op = '*')
        def _backward():
            self.grad += other.data*out.grad
            other.grad += self.data*out.grad
        out._backward = _backward
        return out
    
    def __rmul__(self, other):
        return self * other
    
    def __neg__(self):
        return self*-1
    
    def __sub__(self, other):
        if (isinstance(other, float) or isinstance(other, int)): other = Value(other)
        return self+(-other)
    
    def __rsub__(self, other):
        return other + (-self)
    
    def __pow__(self, other):
        assert (isinstance(other, float) or isinstance(other, int))
        out = self.data**other
        out = Value(out, _prev=(self,), _op = f'**{other}')
        def _backward():
            self.grad += other*(self.data**(other - 1))*out.grad
        out._backward = _backward
        return out
    
    def __truediv__(self, other):
        if (isinstance(other, float) or isinstance(other, int)): other = Value(other)
        """
        a/b = a*(1/b) = a*(b**-1)
        """
        out = self*(other**-1)
        return out
    
    def exp(self):
        out = Value(math.exp(self.data), _prev=(self,), _op = "exp")
        def _backward():
            self.grad += out.data*out.grad
        out._backward = _backward
        return out
    
    def tanh(self):
        """
        tanh(x) = (e^2x -1)/(e^2x + 1)
        tanh'(x) = 1 - tanh^2(x)
        """
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, _prev=(self,), _op="tanh")
        def _backward():
            self.grad  += (1 - out.data**2)*out.grad
        out._backward = _backward
        return out

We need some utility functions to look at the compute graph created

In [178]:
def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root, format='svg', rankdir='LR'):
    """
    format: png | svg | ...
    rankdir: TB (top to bottom graph) | LR (left to right)
    """
    assert rankdir in ['LR', 'TB']
    nodes, edges = trace(root)
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir}) #, node_attr={'rankdir': 'TB'})
    
    for n in nodes:
        dot.node(name=str(id(n)), label = f"label {n.label} | data {n.data} | grad {n.grad}" , shape='record')
        if n._op:
            dot.node(name=str(id(n)) + n._op, label=n._op)
            dot.edge(str(id(n)) + n._op, str(id(n)))
    
    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    
    return dot

In [179]:
a = Value(3.0)
a.label = 'a'
b = Value(2.0)
b.label = 'b'
c = b+a
c.label = 'c'
f = Value(-1)
f.label = 'f'
d = f*c
d.label = 'd'
e = d.tanh()
e.label = 'e'
print(e)

Value(data = -0.9999092042625951)


In [180]:
draw_dot(e)

Error: bad label format label e | data -0.9999092042625951 | grad <function Value.tanh.<locals>._backward at 0x10955ed40>


CalledProcessError: Command '[PosixPath('dot'), '-Kdot', '-Tsvg']' returned non-zero exit status 1. [stderr: 'Error: bad label format label e | data -0.9999092042625951 | grad <function Value.tanh.<locals>._backward at 0x10955ed40>\n']

In [120]:
"""
gradient of d wrt c is c.grad
dd/dc = d(c*f)/dc = f
"""
d.grad = 1.0
c.grad = -1.0
f.grad = 5.0

In [122]:
"""
gradient of d wrt b id b.grad
using the chain rule 
dd/db = dd/dc*dc/db
= c.grad * d(a+b)/db = c.grad -> gradient just passes through
"""
b.grad = -1.0
a.grad = -1.0

In [ ]:
# since the gradient of d wrt c is negative small change in c in positive direction shoul reduce the value of d
# and the opposite for the f as well
def lol():
    h = 0.01
    a = Value(3.0)
    a.label = 'a'
    b = Value(2.0)
    b.label = 'b'
    c = b+a
    c.label = 'c'
    # c.data += h
    f = Value(-1)
    f.label = 'f'
    f.data += h
    d = f*c
    d.label = 'd'
    print(d)

lol()

Value(data = -4.95)
